In [ ]:

import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import List, Dict, Tuple, Optional


class CrossEncoderReranker(nn.Module):
    """
    Cross-encoder reranker for the second stage of a
    retrieve-and-rerank pipeline.

    Takes the candidate pool produced by bi-encoder retrieval
    (Section 5.2) and produces a refined relevance score for
    each user-item pair using full joint encoding.

    Architecture: concatenate user and item feature vectors,
    pass through a deep interaction network.
    This is simpler than a transformer cross-encoder but captures
    the key benefit: direct feature interaction between user and item.
    For transformer-based cross-encoders, see Chapter 6.
    """


    def __init__(
        self,
        user_feature_dim: int,
        item_feature_dim: int,
        hidden_dims: List[int] = [512, 256, 128],
        dropout: float = 0.1,
    ):
        super().__init__()

        # Input: concatenation of user and item features
        input_dim = user_feature_dim + item_feature_dim

        # Build deep interaction network
        layers = []
        current_dim = input_dim

        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(current_dim, hidden_dim),
                nn.LayerNorm(hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout),
            ])
            current_dim = hidden_dim

        # Final scoring layer
        layers.append(nn.Linear(current_dim, 1))

        self.interaction_network = nn.Sequential(*layers)

        # Feature crossing layer — explicit pairwise interactions
        # between user and item features before the MLP
        self.feature_crossing = FeatureCrossingLayer(
            user_feature_dim, item_feature_dim
        )

    def forward(
        self,
        user_features: torch.Tensor,   # (batch_size, user_feature_dim)
        item_features: torch.Tensor,   # (batch_size, num_candidates, item_feature_dim)
    ) -> torch.Tensor:
        """
        Score all candidates for each user in the batch.

        Args:
            user_features: User representation for each query
            item_features: All candidate item features

        Returns:
            scores: (batch_size, num_candidates) relevance scores
        """
        batch_size, num_candidates, item_dim = item_features.shape

        # Expand user features to match candidate dimension
        # (batch_size, user_dim) → (batch_size, num_candidates, user_dim)
        user_expanded = user_features.unsqueeze(1).expand(
            -1, num_candidates, -1
        )

        # Compute explicit feature crosses
        crossed = self.feature_crossing(user_expanded, item_features)
        # (batch_size, num_candidates, cross_dim)

        # Concatenate user features, item features, and crosses
        combined = torch.cat([user_expanded, item_features, crossed], dim=-1)
        # (batch_size, num_candidates, user_dim + item_dim + cross_dim)

        # Reshape for batch processing through the MLP
        combined_flat = combined.view(batch_size * num_candidates, -1)
        scores_flat = self.interaction_network(combined_flat)
        scores = scores_flat.view(batch_size, num_candidates)

        return scores


class FeatureCrossingLayer(nn.Module):
    """
    Computes explicit pairwise interactions between
    user and item features.

    This captures interactions like:
    "user prefers long articles" × "item article length"
    "user is on mobile" × "item has mobile-optimized format"

    These interactions are invisible to a bi-encoder that
    encodes user and item independently.
    """

    def __init__(
        self,
        user_dim: int,
        item_dim: int,
        num_crosses: int = 16,
    ):
        super().__init__()
        # Project user and item to same dimension for crossing
        self.user_proj = nn.Linear(user_dim, num_crosses)
        self.item_proj = nn.Linear(item_dim, num_crosses)

    def forward(
        self,
        user_features: torch.Tensor,  # (B, N, user_dim)
        item_features: torch.Tensor,  # (B, N, item_dim)
    ) -> torch.Tensor:
        # Project to crossing space
        u = self.user_proj(user_features)  # (B, N, num_crosses)
        v = self.item_proj(item_features)  # (B, N, num_crosses)

        # Element-wise product = feature interaction
        crosses = u * v  # (B, N, num_crosses)

        return crosses


class RetrieveAndRerankPipeline:
    """
    Full retrieve-and-rerank pipeline combining:
    1. Bi-encoder retrieval via ANN index (Section 5.2)
    2. Cross-encoder reranking for precision

    This is the production architecture that bridges
    Section 5.2 (retrieval) with Section 5.3 (similarity learning).
    """

    def __init__(
        self,
        bi_encoder_model,           # Two-tower model from Chapter 3 / Section 5.3.3
        retrieval_index,            # ANNRetrievalIndex from Section 5.2.2
        cross_encoder: CrossEncoderReranker,
        item_feature_store,         # Lookup: item_id → feature vector
        retrieval_k: int = 500,     # Candidates from bi-encoder
        final_k: int = 50,          # Items returned after reranking
    ):
        self.bi_encoder = bi_encoder_model
        self.retrieval_index = retrieval_index
        self.cross_encoder = cross_encoder
        self.item_feature_store = item_feature_store
        self.retrieval_k = retrieval_k
        self.final_k = final_k

    def recommend(
        self,
        user_features: torch.Tensor,    # (user_feature_dim,)
        return_scores: bool = False,
    ) -> List[str]:
        """
        Full retrieve-and-rerank recommendation for a single user.

        Stage 1: Bi-encoder retrieval (~5ms)
        Stage 2: Cross-encoder reranking (~50ms)
        """
        # ── Stage 1: Bi-Encoder Retrieval ─────────────────────────────────
        with torch.no_grad():
            user_embedding = self.bi_encoder.user_tower(
                user_features.unsqueeze(0)
            )

        user_np = user_embedding.cpu().numpy().astype("float32")
        candidate_ids, retrieval_scores = self.retrieval_index.query(
            user_np, k=self.retrieval_k
        )

        # ── Stage 2: Cross-Encoder Reranking ──────────────────────────────
        # Fetch item features for all candidates from feature store
        candidate_features = torch.stack([
            self.item_feature_store[item_id]
            for item_id in candidate_ids
        ])  # (num_candidates, item_feature_dim)

        with torch.no_grad():
            rerank_scores = self.cross_encoder(
                user_features.unsqueeze(0),              # (1, user_dim)
                candidate_features.unsqueeze(0),         # (1, num_candidates, item_dim)
            ).squeeze(0)  # (num_candidates,)

        # Sort by reranker score and return top-k
        top_k_indices = torch.argsort(rerank_scores, descending=True)[:self.final_k]
        top_k_item_ids = [candidate_ids[i] for i in top_k_indices.tolist()]

        if return_scores:
            top_k_scores = rerank_scores[top_k_indices].tolist()
            return top_k_item_ids, top_k_scores

        return top_k_item_ids
